<a href="https://colab.research.google.com/github/KarunyaVarma/Karunya-Varma/blob/main/DNN_GTPP_Regression1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# -------------------------------------------------
# 1. Input data & Reproducibility
# -------------------------------------------------

tf.keras.utils.set_random_seed(42)

# Read the CSV file uploaded to Google Colab
data = pd.read_csv('/content/GTPP-DATASET.csv')

# 8 input features
X = data[[
    'AT',
    'AP',
    'DF',
    'CH',
    'GP',
    'CPR',
    'CPD',
    'TTXM'
]].values

# Target Output
y = data['EP'].values


# -------------------------------------------------
# 2. Train-Test Split (80:20 Ratio)
# -------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# -------------------------------------------------
# 3. MinMax Scaling
# -------------------------------------------------

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------------
# 4. Phase 1 Best Settings & Layer Configurations
# -------------------------------------------------

BEST_LR = 0.002
BEST_BS = 16
FIXED_EPOCHS = 100

# Simple, separate configuration list defining the neuron counts
layer_configurations = [
    [128],               # Strategy 1: 1 Layer
    [128, 32],           # Strategy 2: 2 Layers
    [128, 64],          # Strategy 3: 2 Layers
    [128, 128],         # Strategy 4: 2 Layers
    [128, 32, 32],       # Strategy 5: 3 Layers
    [128, 64, 32],      # Strategy 6: 3 Layers
    [128, 128, 32],      # Strategy 7: 3 Layers
    [128, 32, 64],      # Strategy 8: 3 Layers
    [128, 64, 64],      # Strategy 9: 3 Layers
    [128, 128, 64],      # Strategy 10: 3 Layers
    [128, 32, 128],      # Strategy 11: 3 Layers
    [128, 64, 128],      # Strategy 12: 3 Layers
    [128, 128, 128]      # Strategy 13: 3 Layers
]

results_list = []

print("Starting Phase 2: Depth & Layer Architecture Experiments...\n")


# -------------------------------------------------
# 5. Loop Through Configurations
# -------------------------------------------------

for config in layer_configurations:

    label = f"{len(config)} Layer(s) {config}"
    print(f"Testing Architecture: {label} ...")

    tf.keras.utils.set_random_seed(42)

    # Build layers based on the separate config list
    if len(config) == 1:
        model = tf.keras.models.Sequential([
            tf.keras.layers.Input(shape=(8,)),
            tf.keras.layers.Dense(config[0], activation='relu'),
            tf.keras.layers.Dense(1)
        ])
    elif len(config) == 2:
        model = tf.keras.models.Sequential([
            tf.keras.layers.Input(shape=(8,)),
            tf.keras.layers.Dense(config[0], activation='relu'),
            tf.keras.layers.Dense(config[1], activation='relu'),
            tf.keras.layers.Dense(1)
        ])
    elif len(config) == 3:
        model = tf.keras.models.Sequential([
            tf.keras.layers.Input(shape=(8,)),
            tf.keras.layers.Dense(config[0], activation='relu'),
            tf.keras.layers.Dense(config[1], activation='relu'),
            tf.keras.layers.Dense(config[2], activation='relu'),
            tf.keras.layers.Dense(1)
        ])

    # Calculate total parameters
    total_params = model.count_params()

    # Compile Model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=BEST_LR),
        loss='mse',
        metrics=['mae']
    )

    # Train Model
    model.fit(
        X_train_scaled,
        y_train,
        epochs=FIXED_EPOCHS,
        batch_size=BEST_BS,
        validation_split=0.2,
        verbose=0
    )

    # Predict on Test Data
    y_pred = model.predict(X_test_scaled, verbose=0).flatten()

    # Calculate Metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    results_list.append({
        "Architecture": label,
        "Total Parameters": total_params,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    })


# -------------------------------------------------
# 6. Display Comparison Results
# -------------------------------------------------

results_dataframe = pd.DataFrame(results_list)

print("\n==========================================================================")
print("                    PHASE 2: NETWORK DEPTH RESULTS                        ")
print("==========================================================================")
print(results_dataframe.to_string(index=False))

Starting Phase 2: Depth & Layer Architecture Experiments...

Testing Architecture: 1 Layer(s) [128] ...
Testing Architecture: 2 Layer(s) [128, 32] ...
Testing Architecture: 2 Layer(s) [128, 64] ...
Testing Architecture: 2 Layer(s) [128, 128] ...
Testing Architecture: 3 Layer(s) [128, 32, 32] ...
Testing Architecture: 3 Layer(s) [128, 64, 32] ...
Testing Architecture: 3 Layer(s) [128, 128, 32] ...
Testing Architecture: 3 Layer(s) [128, 32, 64] ...
Testing Architecture: 3 Layer(s) [128, 64, 64] ...
Testing Architecture: 3 Layer(s) [128, 128, 64] ...
Testing Architecture: 3 Layer(s) [128, 32, 128] ...
Testing Architecture: 3 Layer(s) [128, 64, 128] ...
Testing Architecture: 3 Layer(s) [128, 128, 128] ...

                    PHASE 2: NETWORK DEPTH RESULTS                        
              Architecture  Total Parameters      MAE      MSE     RMSE  R2 Score
          1 Layer(s) [128]              1281 1.048892 1.872188 1.368279  0.952299
      2 Layer(s) [128, 32]              5313 1.01